# Example outputs for Jill, v70

The three answers whose shape changed today, run as they would run, so the output can be checked before anything is fed to CRIS. Each section: the question, what was agreed, the query.


## SP_7  Season premiere, one row

**How did the Season premiere of Hist Greatest Machines perform?**

Agreed: The first telecast of the latest season against the network's latest-12-month premiere norm, the same norm 2.2 already builds. One row: Network, Program, Season, Premiere Episode, Air Date, Premiere AA, Norm dates, Premiere Norm, % vs norm. Expected: AERIAL ATTACKERS, 1 Jun 2026, 266.


In [ ]:
%%sql
SET p_network       = 'HIST';
SET p_program       = 'HIST GREATEST MACHINES';
SET p_season        = NULL;
SET p_stream        = 'Live+SD';
SET p_demo          = 'DEMO_AA';
SET p_include_specials = FALSE;
SET p_win_from      = NULL;
SET p_win_to        = NULL;
SET p_norm_from     = NULL;
SET p_norm_to       = NULL;
SET p_norm_start_2959 = 2000;
SET p_norm_end_2959   = 2400;
SET win_to   = (SELECT COALESCE(TO_DATE($p_win_to),
                  (SELECT MAX(LATEST_FULL_SUNDAY) FROM AUDIENCE_DEV_DB.CRIS.CRIS_LATEST_DATE
                   WHERE ($p_network IS NULL OR NETWORK_CODE = $p_network)
                     AND RATING_SOURCE = $p_stream)));
SET win_from = (SELECT COALESCE(TO_DATE($p_win_from),
                  DATEADD('day', 1, DATEADD('month', -12, $win_to))));
SET norm_to   = (SELECT COALESCE(TO_DATE($p_norm_to),   $win_to));
SET norm_from = (SELECT COALESCE(TO_DATE($p_norm_from), $win_from));
SET p_premiere_only = TRUE;
SET p_exclude_curse = TRUE;
SET p_movies        = 'EXCLUDE';

SET demo_used = (SELECT IFF(UPPER(COALESCE($p_demo,'DEMO_AA')) = 'DEMO_AA',
                     CASE $p_network
                          WHEN 'AEN'  THEN 'P25_64_AA_ESTIMATES'
                          WHEN 'HIST' THEN 'P25_64_AA_ESTIMATES'
                          WHEN 'FYI'  THEN 'P25_64_AA_ESTIMATES'
                          WHEN 'LIF'  THEN 'F25_64_AA_ESTIMATES'
                          WHEN 'LMN'  THEN 'F25_64_AA_ESTIMATES'
                          WHEN 'VICE' THEN 'P18_49_AA_ESTIMATES'
                          ELSE 'UNRESOLVED - no default demo outside the six'
                     END,
                     UPPER($p_demo)));

WITH
fy AS (
    SELECT FISCAL_YEAR, FY_START, FY_END
    FROM AUDIENCE_DEV_DB.CRIS.CRIS_FY_CALENDAR
),
fact AS (
    SELECT f.*, IDENTIFIER($p_demo) AS DEMO
    FROM AUDIENCE_DEV_DB.CRIS.CRIS_TELECAST_FACT f
    WHERE ($p_network IS NULL OR f.NETWORK_CODE = $p_network)
      AND f.RATING_SOURCE = $p_stream
      AND f.BROADCAST_DATE <= $win_to
      AND ( $p_movies = 'BOTH'
         OR ($p_movies = 'EXCLUDE' AND f.MOVIE_IND = 0)
         OR ($p_movies = 'ONLY'    AND f.MOVIE_IND = 1) )
      AND ($p_include_specials OR f.CONTENT_TYPE <> 'Special')
),
target AS (
    SELECT f2.NETWORK_CODE, f2.PROGRAM_CODE, f2.CT_SEASON,
           IFF(REGEXP_LIKE(f2.CT_SEASON, '^(19|20)[0-9]{2}[A-Za-z]?$'),
               'FY' || fy.FISCAL_YEAR, f2.CT_SEASON)                 AS NORM_KEY,
           REGEXP_LIKE(f2.CT_SEASON, '^(19|20)[0-9]{2}[A-Za-z]?$')   AS IS_YEAR_LABEL,
           fy.FY_START, fy.FY_END
    FROM fact f2
    LEFT JOIN fy
      ON f2.BROADCAST_DATE BETWEEN fy.FY_START AND fy.FY_END
    WHERE f2.IS_FIRST_RUN_CT
      AND ($p_program IS NULL OR f2.PROGRAM_NAME ILIKE $p_program)
      AND ($p_season  IS NULL OR f2.CT_SEASON = $p_season)
      AND f2.CT_SEASON IS NOT NULL
    QUALIFY ROW_NUMBER() OVER (PARTITION BY f2.NETWORK_CODE, f2.PROGRAM_CODE
                               ORDER BY f2.BROADCAST_DATE DESC, f2.TELECAST_NUM DESC) = 1
),
series AS (
    SELECT f.*, t.NORM_KEY, t.IS_YEAR_LABEL
    FROM fact f
    JOIN target t
      ON  t.NETWORK_CODE = f.NETWORK_CODE
      AND t.PROGRAM_CODE = f.PROGRAM_CODE
      AND ( (NOT t.IS_YEAR_LABEL AND f.CT_SEASON = t.CT_SEASON)
         OR (t.IS_YEAR_LABEL
             AND f.BROADCAST_DATE BETWEEN t.FY_START AND t.FY_END) )
    WHERE f.IS_FIRST_RUN_CT
),
net_norm AS (
    SELECT NETWORK_CODE,
           SUM(IDENTIFIER($p_demo) * HH_MIN)
             / NULLIF(SUM(HH_MIN), 0)                    AS NET_PREM_AA,
           SUM(HH_MIN)                                   AS NET_PREM_MIN,
           COUNT(DISTINCT BROADCAST_DATE)                AS NET_NIGHTS,
           COUNT(DISTINCT PROGRAM_CODE || '|' || TELECAST_NUM
                          || '|' || BROADCAST_DATE)      AS NET_TCASTS,
           COUNT(DISTINCT PROGRAM_CODE)                  AS NET_PROGRAMMES
    FROM AUDIENCE_DEV_DB.CRIS.CRIS_HALFHR_FACT
    WHERE IS_AE_NETWORK
      AND ($p_network IS NULL OR NETWORK_CODE = $p_network)
      AND RATING_SOURCE = $p_stream
      AND IS_FIRST_RUN_CT
      AND BROADCAST_DATE BETWEEN $norm_from AND $norm_to
      AND ($p_norm_start_2959 IS NULL OR HH_START_2959 >= $p_norm_start_2959)
      AND ($p_norm_end_2959   IS NULL OR HH_START_2959 <  $p_norm_end_2959)
      AND ( $p_movies = 'BOTH'
         OR ($p_movies = 'EXCLUDE' AND MOVIE_IND = 0)
         OR ($p_movies = 'ONLY'    AND MOVIE_IND = 1) )
      AND ($p_include_specials OR CONTENT_TYPE <> 'Special')
      AND NOT ($p_exclude_curse AND NETWORK_CODE = 'HIST'
               AND UPPER(PROGRAM_NAME) LIKE '%CURSE%OAK ISLAND%')
    GROUP BY NETWORK_CODE
),
measured AS (
    SELECT *
    FROM series
    WHERE NOT $p_premiere_only OR IS_PREMIERE_CT = 1
),
ser AS (
    SELECT NETWORK_CODE, PROGRAM_CODE, MAX(PROGRAM_NAME) AS PROGRAM_NAME,
           MAX(IFF($p_premiere_only, EPISODE_NAME, NULL)) AS PREMIERE_EPISODE,
           NORM_KEY,
           COUNT(*) AS TCASTS,
           MIN(BROADCAST_DATE) AS FROM_DATE,
           MAX(BROADCAST_DATE) AS TO_DATE,
           SUM(DEMO * CONTRIBUTING_DURATION) / NULLIF(SUM(CONTRIBUTING_DURATION), 0) AS SERIES_AA
    FROM measured
    GROUP BY NETWORK_CODE, PROGRAM_CODE, NORM_KEY
)
SELECT
    s.NETWORK_CODE  AS NETWORK,
    s.PROGRAM_NAME  AS PROGRAM,
    s.NORM_KEY      AS SEASON,
    IFF($p_premiere_only, 'SEASON PREMIERE', 'SEASON')             AS MEASURED,
    s.PREMIERE_EPISODE,
    s.FROM_DATE                                                    AS AIR_DATE,
    s.TO_DATE, s.TCASTS,
    REPLACE($demo_used, '_AA_ESTIMATES', '') AS DEMO_USED,
    ROUND(s.SERIES_AA/1000, 0)                                     AS SERIES_AA_000,
    $norm_from AS NORM_FROM, $norm_to AS NORM_TO,
    ROUND(n.NET_PREM_AA/1000, 0)                                   AS PREMIERE_NORM_000,
    ROUND((ROUND(s.SERIES_AA/1000,0)
           / NULLIF(ROUND(n.NET_PREM_AA/1000,0),0) - 1) * 100, 1)  AS PCT_VS_NORM,
    ROUND(s.SERIES_AA/1000, 0) - ROUND(n.NET_PREM_AA/1000, 0)      AS DIFF_000,
    CASE WHEN ROUND(s.SERIES_AA/1000,0) > ROUND(n.NET_PREM_AA/1000,0) THEN 'ABOVE'
         WHEN ROUND(s.SERIES_AA/1000,0) < ROUND(n.NET_PREM_AA/1000,0) THEN 'BELOW'
         ELSE 'IN LINE' END                                        AS VERDICT,
    n.NET_TCASTS AS NORM_TCASTS, n.NET_PROGRAMMES AS NORM_PROGRAMMES
FROM ser s
JOIN net_norm n ON n.NETWORK_CODE = s.NETWORK_CODE
ORDER BY SERIES_AA_000 DESC;

## SP_IND_10  Season quarter hours against the time-period premiere norm

**What is the quarter-hour build/decline within Secret Skinwalker Ranch's telecasts vs. the season QH norm?**

Agreed: Four rows for the season: QH1 to QH4 averaged across every telecast, with QH1 to QH4 movement. The norm is the latest 12 months of HIST first-run premieres airing in the same time period as Skinwalker, 9-10pm, all days of the week, position by position, with no specials, mini-series or Curse exclusions.


In [ ]:
%%sql
SET p_network       = 'HIST';
SET p_program       = 'SECRET SKINWALKER RANCH';
SET p_stream        = 'Live+SD';
SET p_norm_basis    = 'TIME_PERIOD';
SET p_norm_months   = 12;
SET p_rollup        = TRUE;
SET p_demo      = 'DEMO_AA';
SET p_season        = NULL;
SET p_runover_max_min = 15;
SET p_win_to   = NULL;
SET win_to = (SELECT COALESCE(TO_DATE($p_win_to),
                (SELECT LATEST_QH_BROADCAST_DATE FROM AUDIENCE_DEV_DB.CRIS.CRIS_LATEST_DATE
                 WHERE NETWORK_CODE = $p_network AND RATING_SOURCE = $p_stream)));

SET demo_used = (SELECT IFF(UPPER(COALESCE($p_demo,'DEMO_AA')) = 'DEMO_AA',
                     CASE $p_network
                          WHEN 'AEN'  THEN 'P25_64_AA_ESTIMATES'
                          WHEN 'HIST' THEN 'P25_64_AA_ESTIMATES'
                          WHEN 'FYI'  THEN 'P25_64_AA_ESTIMATES'
                          WHEN 'LIF'  THEN 'F25_64_AA_ESTIMATES'
                          WHEN 'LMN'  THEN 'F25_64_AA_ESTIMATES'
                          WHEN 'VICE' THEN 'P18_49_AA_ESTIMATES'
                          ELSE 'UNRESOLVED - no default demo outside the six'
                     END,
                     UPPER($p_demo)));

WITH base AS (
    SELECT f.NETWORK_CODE, f.PROGRAM_CODE, f.TELECAST_NUM,
           f.PROGRAM_NAME, f.EPISODE_NAME, f.CT_SEASON,
           f.BROADCAST_DATE, f.QTR_START_TIME, f.QH_START_2959, f.EP_START_2959,
           CAST(f.QH_MIN AS INT) AS QH_MIN,
           IDENTIFIER($p_demo) AS DEMO
    FROM AUDIENCE_DEV_DB.CRIS.CRIS_QH_FACT f
    WHERE f.NETWORK_CODE  = $p_network
      AND f.RATING_SOURCE = $p_stream
      AND f.IS_FIRST_RUN_CT
      AND f.PROGRAM_NAME ILIKE $p_program
      AND f.BROADCAST_DATE <= $win_to
      AND ($p_season IS NULL OR f.CT_SEASON = $p_season)
      AND f.CT_SEASON IS NOT NULL),
scoped AS (
    SELECT b.* FROM base b
    JOIN (SELECT PROGRAM_CODE, CT_SEASON
          FROM base
          QUALIFY ROW_NUMBER() OVER (PARTITION BY PROGRAM_CODE
                                     ORDER BY BROADCAST_DATE DESC) = 1) t
      ON t.PROGRAM_CODE = b.PROGRAM_CODE
     AND EQUAL_NULL(t.CT_SEASON, b.CT_SEASON)
),
seq AS (
    SELECT s.*,
        ROW_NUMBER() OVER (PARTITION BY PROGRAM_CODE, TELECAST_NUM
                           ORDER BY QH_START_2959)  AS QH_NUM,
        COUNT(*)     OVER (PARTITION BY PROGRAM_CODE, TELECAST_NUM) AS TOTAL_QH
    FROM scoped s
),
kept AS (
    SELECT * FROM seq
    WHERE NOT (QH_NUM = TOTAL_QH AND QH_MIN < $p_runover_max_min)
),
tp_norm AS (
    SELECT n.QH_NUM,
           SUM(n.DEMO * n.QH_MIN) / NULLIF(SUM(n.QH_MIN), 0) AS TP_NORM_AA,
           COUNT(DISTINCT n.PROGRAM_CODE || '|' || n.TELECAST_NUM || '|' || n.BROADCAST_DATE) AS TP_NORM_TC
    FROM (
        SELECT f.PROGRAM_CODE, f.TELECAST_NUM, f.BROADCAST_DATE, f.EP_START_2959,
               CAST(f.QH_MIN AS INT) AS QH_MIN, IDENTIFIER($p_demo) AS DEMO,
               ROW_NUMBER() OVER (PARTITION BY f.PROGRAM_CODE, f.TELECAST_NUM
                                  ORDER BY f.QH_START_2959) AS QH_NUM,
               COUNT(*) OVER (PARTITION BY f.PROGRAM_CODE, f.TELECAST_NUM) AS TOTAL_QH
        FROM AUDIENCE_DEV_DB.CRIS.CRIS_QH_FACT f
        WHERE f.NETWORK_CODE  = $p_network
          AND f.RATING_SOURCE = $p_stream
          AND f.IS_FIRST_RUN_CT
          AND f.BROADCAST_DATE BETWEEN DATEADD('day', 1, DATEADD('month', -$p_norm_months, $win_to))
                                   AND $win_to
    ) n
    JOIN (SELECT DISTINCT FLOOR(EP_START_2959 / 100) * 100 + IFF(MOD(EP_START_2959, 100) >= 30, 30, 0) AS START_HH,
                          TOTAL_QH
          FROM seq) shape
      ON  FLOOR(n.EP_START_2959 / 100) * 100 + IFF(MOD(n.EP_START_2959, 100) >= 30, 30, 0) = shape.START_HH
      AND n.TOTAL_QH = shape.TOTAL_QH
    WHERE NOT (n.QH_NUM = n.TOTAL_QH AND n.QH_MIN < $p_runover_max_min)
    GROUP BY n.QH_NUM
),
normed AS (
    SELECT k.*,
        t.TP_NORM_AA, t.TP_NORM_TC,
        IFF($p_norm_basis = 'TIME_PERIOD', t.TP_NORM_AA,
            (SUM(k.DEMO * k.QH_MIN) OVER (PARTITION BY k.PROGRAM_CODE, k.CT_SEASON, k.QH_NUM)
               - k.DEMO * k.QH_MIN)
              / NULLIF(SUM(k.QH_MIN) OVER (PARTITION BY k.PROGRAM_CODE, k.CT_SEASON, k.QH_NUM)
                       - k.QH_MIN, 0))                                 AS NORM_AA,
        IFF($p_norm_basis = 'TIME_PERIOD', t.TP_NORM_TC,
            COUNT(*) OVER (PARTITION BY k.PROGRAM_CODE, k.CT_SEASON, k.QH_NUM) - 1) AS NORM_TC,
        LAG(k.DEMO) OVER (PARTITION BY k.PROGRAM_CODE, k.TELECAST_NUM
                        ORDER BY k.QH_NUM)                             AS PREV_AA,
        FIRST_VALUE(k.DEMO) OVER (PARTITION BY k.PROGRAM_CODE, k.TELECAST_NUM
                        ORDER BY k.QH_NUM
                        ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING) AS QH1_AA,
        LAST_VALUE(k.DEMO)  OVER (PARTITION BY k.PROGRAM_CODE, k.TELECAST_NUM
                        ORDER BY k.QH_NUM
                        ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING) AS LAST_AA
    FROM kept k
    LEFT JOIN tp_norm t ON t.QH_NUM = k.QH_NUM
),
season_qh AS (
    SELECT NETWORK_CODE, PROGRAM_NAME, CT_SEASON, QH_NUM,
           SUM(DEMO * QH_MIN)    / NULLIF(SUM(QH_MIN), 0)  AS QH_AA,
           SUM(NORM_AA * QH_MIN) / NULLIF(SUM(QH_MIN), 0)  AS QH_NORM,
           MAX(NORM_TC)                                    AS QH_NORM_TC,
           SUM(QH_MIN)                                     AS QH_MIN,
           COUNT(*)                                        AS TELECASTS
    FROM normed
    WHERE $p_rollup
    GROUP BY NETWORK_CODE, PROGRAM_NAME, CT_SEASON, QH_NUM
),
rolled AS (
    SELECT s.*,
           LAG(QH_AA) OVER (PARTITION BY NETWORK_CODE, PROGRAM_NAME, CT_SEASON ORDER BY QH_NUM) AS PREV_AA,
           FIRST_VALUE(QH_AA) OVER (PARTITION BY NETWORK_CODE, PROGRAM_NAME, CT_SEASON ORDER BY QH_NUM)  AS QH1_AA,
           LAST_VALUE(QH_AA)  OVER (PARTITION BY NETWORK_CODE, PROGRAM_NAME, CT_SEASON ORDER BY QH_NUM
                                    ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING) AS LAST_AA
    FROM season_qh s
)
SELECT
    REPLACE($demo_used, '_AA_ESTIMATES', '') AS DEMO_USED,
    NETWORK_CODE   AS NETWORK,
    BROADCAST_DATE AS AIR_DATE,
    PROGRAM_NAME   AS PROGRAM,
    EPISODE_NAME   AS EPISODE,
    CT_SEASON     AS SEASON,
    'QH' || QH_NUM AS QUARTER_HOUR,
    QTR_START_TIME AS QH_START,
    QH_MIN,
    ROUND(DEMO/1000, 0)                                            AS AA_000,
    CASE WHEN PREV_AA IS NOT NULL
         THEN ROUND((ROUND(DEMO/1000,0)/NULLIF(ROUND(PREV_AA/1000,0),0)-1)*100, 1) END
                                                                   AS VS_PRIOR_QH_PCT,
    ROUND(NORM_AA/1000, 0)                                         AS NORM_AA_000,
    NORM_TC,
    ROUND((ROUND(DEMO/1000,0)/NULLIF(ROUND(NORM_AA/1000,0),0)-1)*100, 1) AS VS_NORM_PCT,
    ROUND((ROUND(LAST_AA/1000,0)/NULLIF(ROUND(QH1_AA/1000,0),0)-1)*100, 1) AS LAST_VS_FIRST_PCT,
    $win_to                 AS TO_DATE
FROM normed
WHERE NOT $p_rollup
UNION ALL
SELECT
    REPLACE($demo_used, '_AA_ESTIMATES', '') AS DEMO_USED,
    NETWORK_CODE   AS NETWORK,
    NULL           AS AIR_DATE,
    PROGRAM_NAME   AS PROGRAM,
    'SEASON AVERAGE ' || TELECASTS || ' TELECASTS' AS EPISODE,
    CT_SEASON      AS SEASON,
    'QH' || QH_NUM AS QUARTER_HOUR,
    NULL           AS QH_START,
    QH_MIN,
    ROUND(QH_AA/1000, 0)                                           AS AA_000,
    CASE WHEN PREV_AA IS NOT NULL
         THEN ROUND((ROUND(QH_AA/1000,0)/NULLIF(ROUND(PREV_AA/1000,0),0)-1)*100, 1) END
                                                                   AS VS_PRIOR_QH_PCT,
    ROUND(QH_NORM/1000, 0)                                         AS NORM_AA_000,
    QH_NORM_TC                                                     AS NORM_TC,
    ROUND((ROUND(QH_AA/1000,0)/NULLIF(ROUND(QH_NORM/1000,0),0)-1)*100, 1) AS VS_NORM_PCT,
    ROUND((ROUND(LAST_AA/1000,0)/NULLIF(ROUND(QH1_AA/1000,0),0)-1)*100, 1) AS LAST_VS_FIRST_PCT,
    $win_to                 AS TO_DATE
FROM rolled
ORDER BY 3 NULLS LAST, 7;

## R_6  Network ranks, full universe, ties on all four ranks

**What is History's rank vs. ad-supported entertainment cable networks across Prime and Total Day? FY26 thru 7/5/26 L+SD**

Agreed: Every cable network comes back, the asked one marked in IS_TARGET; the written answer focuses on it. Ties on current and year-ago, all cable and entertainment cable, shown as 7 (tied). The entertainment-cable rank is NULL for news and sports networks, which rank among all cable only.


In [ ]:
%%sql
SET p_network       = 'HIST';
SET p_fiscal_year   = 2026;
SET p_fy_to         = '2026-07-05';
SET p_stream        = 'Live+SD';
SET p_demo          = 'P25_64_AA_ESTIMATES';
SET p_weight_col    = 'HH_MIN';
SET p_excl_by       = 'CODE';
SET data_max = (SELECT MAX(BROADCAST_DATE) FROM AUDIENCE_DEV_DB.CRIS.CRIS_HALFHR_FACT
                WHERE RATING_SOURCE = $p_stream);
SET fy = (SELECT COALESCE($p_fiscal_year,
            (SELECT MAX(FISCAL_YEAR) FROM AUDIENCE_DEV_DB.CRIS.CRIS_FY_CALENDAR
             WHERE FY_START <= $data_max)));
SET fy_start   = (SELECT FY_START FROM AUDIENCE_DEV_DB.CRIS.CRIS_FY_CALENDAR WHERE FISCAL_YEAR = $fy);
SET fy_to      = (SELECT LEAST(COALESCE(TO_DATE($p_fy_to), $data_max), COALESCE(
                    (SELECT FY_END FROM AUDIENCE_DEV_DB.CRIS.CRIS_FY_CALENDAR WHERE FISCAL_YEAR = $fy),
                    $data_max)));
SET pfy_start  = (SELECT PRIOR_FY_START FROM AUDIENCE_DEV_DB.CRIS.CRIS_FY_CALENDAR WHERE FISCAL_YEAR = $fy);
SET pfy_to     = (SELECT DATEADD('day', DATEDIFF('day', $fy_start, $fy_to), $pfy_start));

SET demo_used = (SELECT IFF(UPPER(COALESCE($p_demo,'DEMO_AA')) = 'DEMO_AA',
                     CASE $p_network
                          WHEN 'AEN'  THEN 'P25_64_AA_ESTIMATES'
                          WHEN 'HIST' THEN 'P25_64_AA_ESTIMATES'
                          WHEN 'FYI'  THEN 'P25_64_AA_ESTIMATES'
                          WHEN 'LIF'  THEN 'F25_64_AA_ESTIMATES'
                          WHEN 'LMN'  THEN 'F25_64_AA_ESTIMATES'
                          WHEN 'VICE' THEN 'P18_49_AA_ESTIMATES'
                          ELSE 'UNRESOLVED - no default demo outside the six'
                     END,
                     UPPER($p_demo)));

WITH dayparts AS (
              SELECT 1 AS ORD, 'PRIME M-SU 8P-11P' AS DAYPART, 2000 AS FROM_2959, 2300 AS TO_2959
    UNION ALL SELECT 2,         'TOTAL DAY 6A-6A',               600,               2960
),
periods AS (
              SELECT 'CURRENT'  AS PERIOD, $fy_start  AS D_FROM, $fy_to  AS D_TO
    UNION ALL SELECT 'YEAR AGO',           $pfy_start,           $pfy_to
),
measured AS (
    SELECT p.PERIOD, d.ORD, d.DAYPART, h.NETWORK_CODE,
           u.IS_CABLE, u.IS_ENT_BY_CODE, u.IS_ENT_BY_NAME,
           SUM(IDENTIFIER($p_demo    ) * COALESCE(IDENTIFIER($p_weight_col), h.HH_MIN))
             / NULLIF(SUM(COALESCE(IDENTIFIER($p_weight_col), h.HH_MIN)), 0) AS AA,
           SUM(h.HH_MIN) AS MINUTES
    FROM AUDIENCE_DEV_DB.CRIS.CRIS_HALFHR_FACT h
    JOIN AUDIENCE_DEV_DB.CRIS.CRIS_NETWORK_UNIVERSE u
      ON u.NETWORK_CODE = h.NETWORK_CODE
    JOIN periods p
      ON h.BROADCAST_DATE BETWEEN p.D_FROM AND p.D_TO
    JOIN dayparts d
      ON h.HH_START_2959 >= d.FROM_2959 AND h.HH_START_2959 < d.TO_2959
    WHERE h.RATING_SOURCE = $p_stream
      AND u.IS_CABLE
      AND NOT u.IS_ARTIFACT
    GROUP BY p.PERIOD, d.ORD, d.DAYPART, h.NETWORK_CODE,
             u.IS_CABLE, u.IS_ENT_BY_CODE, u.IS_ENT_BY_NAME
),
ranked AS (
    SELECT m.*,
        DENSE_RANK() OVER (PARTITION BY PERIOD, ORD
                     ORDER BY ROUND(AA/1000, 0) DESC)          AS RNK_ALL_CABLE,
        DENSE_RANK() OVER (PARTITION BY PERIOD, ORD,
                     IFF( ($p_excl_by = 'CODE' AND IS_ENT_BY_CODE)
                       OR ($p_excl_by = 'NAME' AND IS_ENT_BY_NAME), 1, 0)
                     ORDER BY ROUND(AA/1000, 0) DESC)          AS RNK_ENT_CABLE,
        COUNT(*) OVER (PARTITION BY PERIOD, ORD,
                     IFF( ($p_excl_by = 'CODE' AND IS_ENT_BY_CODE)
                       OR ($p_excl_by = 'NAME' AND IS_ENT_BY_NAME), 1, 0),
                     ROUND(AA/1000, 0)) > 1                    AS TIED_ENT_CABLE,
        COUNT(*) OVER (PARTITION BY PERIOD, ORD, ROUND(AA/1000, 0)) > 1 AS TIED_ALL_CABLE,
        COUNT(*) OVER (PARTITION BY PERIOD, ORD)               AS N_ALL_CABLE,
        COUNT_IF( ($p_excl_by = 'CODE' AND IS_ENT_BY_CODE)
               OR ($p_excl_by = 'NAME' AND IS_ENT_BY_NAME) )
            OVER (PARTITION BY PERIOD, ORD)                     AS N_ENT_CABLE
    FROM measured m
),
net AS (
    SELECT * FROM ranked
)
SELECT
    REPLACE($demo_used, '_AA_ESTIMATES', '') AS DEMO_USED,
    c.DAYPART,
    c.NETWORK_CODE                                             AS NETWORK,
    $fy                                                        AS FISCAL_YEAR,
    $fy_start AS FY_FROM, $fy_to AS FY_TO,
    $pfy_start AS PRIOR_FROM, $pfy_to AS PRIOR_TO,
    ROUND(c.AA/1000, 0)                                        AS AA_000,
    IFF($p_network IS NOT NULL AND c.NETWORK_CODE = $p_network, 1, 0) AS IS_TARGET,
    c.RNK_ALL_CABLE                                            AS RANK_ALL_CABLE,
    c.TIED_ALL_CABLE                                           AS TIED_ALL_CABLE,
    c.RNK_ALL_CABLE || IFF(c.TIED_ALL_CABLE, ' (tied)', '')    AS RANK_ALL_CABLE_LABEL,
    IFF( ($p_excl_by = 'CODE' AND c.IS_ENT_BY_CODE)
      OR ($p_excl_by = 'NAME' AND c.IS_ENT_BY_NAME), TRUE, FALSE)  AS IS_ENT_CABLE,
    IFF( ($p_excl_by = 'CODE' AND c.IS_ENT_BY_CODE)
      OR ($p_excl_by = 'NAME' AND c.IS_ENT_BY_NAME), c.RNK_ENT_CABLE, NULL) AS RANK_ENT_CABLE,
    IFF( ($p_excl_by = 'CODE' AND c.IS_ENT_BY_CODE)
      OR ($p_excl_by = 'NAME' AND c.IS_ENT_BY_NAME), c.TIED_ENT_CABLE, NULL) AS TIED,
    IFF( ($p_excl_by = 'CODE' AND c.IS_ENT_BY_CODE)
      OR ($p_excl_by = 'NAME' AND c.IS_ENT_BY_NAME),
        c.RNK_ENT_CABLE || IFF(c.TIED_ENT_CABLE, ' (tied)', ''), NULL) AS RANK_ENT_CABLE_LABEL,
    ROUND(y.AA/1000, 0)                                        AS AA_000_YEAR_AGO,
    y.RNK_ALL_CABLE                                            AS RANK_ALL_CABLE_YEAR_AGO,
    y.RNK_ALL_CABLE || IFF(y.TIED_ALL_CABLE, ' (tied)', '')    AS RANK_ALL_CABLE_YEAR_AGO_LABEL,
    IFF( ($p_excl_by = 'CODE' AND y.IS_ENT_BY_CODE)
      OR ($p_excl_by = 'NAME' AND y.IS_ENT_BY_NAME), y.RNK_ENT_CABLE, NULL) AS RANK_ENT_CABLE_YEAR_AGO,
    IFF( ($p_excl_by = 'CODE' AND y.IS_ENT_BY_CODE)
      OR ($p_excl_by = 'NAME' AND y.IS_ENT_BY_NAME),
        y.RNK_ENT_CABLE || IFF(y.TIED_ENT_CABLE, ' (tied)', ''), NULL) AS RANK_ENT_CABLE_YEAR_AGO_LABEL,
    ROUND((ROUND(c.AA/1000,0)
           / NULLIF(ROUND(y.AA/1000,0),0) - 1) * 100, 1)       AS PCT_VS_YEAR_AGO,
    c.N_ALL_CABLE                                              AS UNIVERSE_ALL_CABLE,
    c.N_ENT_CABLE                                              AS UNIVERSE_ENT_CABLE,
    $p_stream AS STREAM, $p_demo     AS DEMO,
    $p_weight_col AS WEIGHTED_ON, $p_excl_by AS EXCLUSION_LIST,
    $fy_to                  AS TO_DATE
FROM net c
LEFT JOIN net y ON y.NETWORK_CODE = c.NETWORK_CODE
               AND y.ORD = c.ORD AND y.PERIOD = 'YEAR AGO'
WHERE c.PERIOD = 'CURRENT'
ORDER BY c.ORD, c.RNK_ALL_CABLE, c.NETWORK_CODE;

## SP_IND_8  Network rank in Skinwalker's time period, confirmed

**Where does HIST rank among networks in Secret Skinwalker Ranch's time period?**

Agreed: Matched Jill's values in the session after the 15-minute cutoff; Season 7 dates confirmed. Included so Hassan has it in the same set.


In [ ]:
%%sql
SET p_network       = 'HIST';
SET p_program       = 'SECRET SKINWALKER RANCH';
SET p_stream        = 'Live+SD';
SET p_slot_min_min  = 15;
SET p_demo      = 'P25_64_AA_ESTIMATES';
SET p_excl_by       = 'CODE';
SET p_season        = NULL;
SET p_win_from      = NULL;
SET p_win_to        = NULL;
SET win_to   = (SELECT COALESCE(TO_DATE($p_win_to),
                  (SELECT MAX(LATEST_FULL_SUNDAY) FROM AUDIENCE_DEV_DB.CRIS.CRIS_LATEST_DATE
                   WHERE NETWORK_CODE = $p_network AND RATING_SOURCE = $p_stream)));
SET tp_program_code = (SELECT PROGRAM_CODE
                       FROM AUDIENCE_DEV_DB.CRIS.CRIS_TELECAST_FACT
                       WHERE ($p_network IS NULL OR NETWORK_CODE = $p_network)
                      AND RATING_SOURCE = $p_stream
                      AND IS_FIRST_RUN_CT
                      AND ($p_program IS NULL OR PROGRAM_NAME ILIKE $p_program)
                      AND CT_SEASON IS NOT NULL
                      AND BROADCAST_DATE <= $win_to
                       ORDER BY BROADCAST_DATE DESC, TELECAST_NUM DESC LIMIT 1);
SET tp_season = (SELECT COALESCE($p_season, CT_SEASON)
                 FROM AUDIENCE_DEV_DB.CRIS.CRIS_TELECAST_FACT
                 WHERE ($p_network IS NULL OR NETWORK_CODE = $p_network)
                      AND RATING_SOURCE = $p_stream
                      AND IS_FIRST_RUN_CT
                      AND ($p_program IS NULL OR PROGRAM_NAME ILIKE $p_program)
                      AND CT_SEASON IS NOT NULL
                      AND BROADCAST_DATE <= $win_to
                   AND PROGRAM_CODE = $tp_program_code
                 ORDER BY BROADCAST_DATE DESC, TELECAST_NUM DESC LIMIT 1);
SET season_from = (SELECT MIN(BROADCAST_DATE)
                   FROM AUDIENCE_DEV_DB.CRIS.CRIS_TELECAST_FACT
                   WHERE ($p_network IS NULL OR NETWORK_CODE = $p_network)
                      AND RATING_SOURCE = $p_stream
                      AND IS_FIRST_RUN_CT
                      AND ($p_program IS NULL OR PROGRAM_NAME ILIKE $p_program)
                      AND CT_SEASON IS NOT NULL
                      AND BROADCAST_DATE <= $win_to
                     AND PROGRAM_CODE = $tp_program_code
                     AND CT_SEASON    = $tp_season);
SET season_to   = (SELECT MAX(BROADCAST_DATE)
                   FROM AUDIENCE_DEV_DB.CRIS.CRIS_TELECAST_FACT
                   WHERE ($p_network IS NULL OR NETWORK_CODE = $p_network)
                      AND RATING_SOURCE = $p_stream
                      AND IS_FIRST_RUN_CT
                      AND ($p_program IS NULL OR PROGRAM_NAME ILIKE $p_program)
                      AND CT_SEASON IS NOT NULL
                      AND BROADCAST_DATE <= $win_to
                     AND PROGRAM_CODE = $tp_program_code
                     AND CT_SEASON    = $tp_season);
SET win_from = (SELECT COALESCE(TO_DATE($p_win_from), $season_from,
                  DATEADD('day', 1, DATEADD('month', -12, $win_to))));
SET win_to   = (SELECT COALESCE(TO_DATE($p_win_to), $season_to, $win_to));

SET demo_used = (SELECT IFF(UPPER(COALESCE($p_demo,'DEMO_AA')) = 'DEMO_AA',
                     CASE $p_network
                          WHEN 'AEN'  THEN 'P25_64_AA_ESTIMATES'
                          WHEN 'HIST' THEN 'P25_64_AA_ESTIMATES'
                          WHEN 'FYI'  THEN 'P25_64_AA_ESTIMATES'
                          WHEN 'LIF'  THEN 'F25_64_AA_ESTIMATES'
                          WHEN 'LMN'  THEN 'F25_64_AA_ESTIMATES'
                          WHEN 'VICE' THEN 'P18_49_AA_ESTIMATES'
                          ELSE 'UNRESOLVED - no default demo outside the six'
                     END,
                     UPPER($p_demo)));

WITH slots AS (
    SELECT BROADCAST_DATE, HALFHR_START_TIME, HH_START_2959
    FROM AUDIENCE_DEV_DB.CRIS.CRIS_HALFHR_FACT
    WHERE NETWORK_CODE  = $p_network
      AND RATING_SOURCE = $p_stream
      AND IS_FIRST_RUN_CT
      AND PROGRAM_NAME ILIKE $p_program
      AND ($p_season IS NULL OR CT_SEASON = $p_season)
      AND BROADCAST_DATE BETWEEN $win_from AND $win_to
    GROUP BY BROADCAST_DATE, HALFHR_START_TIME, HH_START_2959
    HAVING ($p_slot_min_min IS NULL OR SUM(HH_MIN) >= $p_slot_min_min)
),
competitive AS (
    SELECT h.NETWORK_CODE,
           COUNT(DISTINCT h.BROADCAST_DATE)                 AS N_DATES,
           COUNT(*)                                         AS N_HALFHRS,
           SUM(h.HH_MIN)                                    AS MINUTES,
           SUM(IDENTIFIER($p_demo) * h.HH_MIN)
             / NULLIF(SUM(h.HH_MIN), 0) / 1000              AS DEMO_AA_000
    FROM AUDIENCE_DEV_DB.CRIS.CRIS_HALFHR_FACT h
    JOIN slots s
      ON  s.BROADCAST_DATE   = h.BROADCAST_DATE
      AND s.HALFHR_START_TIME = h.HALFHR_START_TIME
    JOIN AUDIENCE_DEV_DB.CRIS.CRIS_NETWORK_UNIVERSE u
      ON u.NETWORK_CODE = h.NETWORK_CODE
    WHERE h.RATING_SOURCE = $p_stream
      AND NOT u.IS_ARTIFACT
      AND ( ($p_excl_by = 'CODE' AND u.IS_ENT_BY_CODE)
         OR ($p_excl_by = 'NAME' AND u.IS_ENT_BY_NAME) )
    GROUP BY h.NETWORK_CODE
),
ranked AS (
    SELECT RANK() OVER (ORDER BY ROUND(DEMO_AA_000, 0) DESC) AS RNK,
           NETWORK_CODE, N_DATES, N_HALFHRS, MINUTES,
           ROUND(DEMO_AA_000, 0) AS AA_000,
           COUNT(*) OVER ()      AS UNIVERSE_SIZE
    FROM competitive
)
SELECT
    REPLACE($demo_used, '_AA_ESTIMATES', '') AS DEMO_USED,
    RNK, UNIVERSE_SIZE, NETWORK_CODE AS NETWORK,
    N_DATES, N_HALFHRS, MINUTES, AA_000,
    IFF(NETWORK_CODE = $p_network, 1, 0)                     AS IS_TARGET,
    IFF(NETWORK_CODE = $p_network, '<<< the series network', NULL) AS MARKER,
    (SELECT COUNT(*) FROM slots)     AS SLOTS_IN_PERIOD,
    (SELECT MIN(BROADCAST_DATE) FROM slots) AS PERIOD_FROM,
    (SELECT MAX(BROADCAST_DATE) FROM slots) AS PERIOD_TO,
    $p_excl_by                              AS EXCLUSION_LIST,
    $win_from               AS FROM_DATE,
    $win_to                 AS TO_DATE
FROM ranked
ORDER BY RNK;